# Notebook 02 — Preprocessing Teks Ulasan EWA

Fase CRISP-DM : Data Preparation
Input : dataset_raw_ewa_acsc.csv (3.628 ulasan)
Output : dataset_clean_ewa_acsc.csv, skip_documentation.csv, translation_audit_log.csv, distribusi_bahasa.csv/.png

Catatan reproduktibilitas: langdetect bersifat non-deterministik, sehingga dikunci dengan DetectorFactory.seed = 0 agar hasil konsisten di setiap run. Jalankan sel dari atas ke bawah setelah menyesuaikan BASE.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Sel 1 — Setup & Muat Data Mentah

In [2]:
# from google.colab import drive; drive.mount('/content/drive')
!pip install -q langdetect deep-translator
import pandas as pd, numpy as np, re, matplotlib.pyplot as plt

BASE = "/content/drive/MyDrive/TA_ACSC_EWA"
RAW_PATH = f"{BASE}/01_scraping/data/dataset_raw_ewa_acsc.csv"
OUT_DIR  = f"{BASE}/02_preprocessing/data"

df = pd.read_csv(RAW_PATH)
print(f"Data mentah: {len(df)} baris")
print("Kolom:", df.columns.tolist())


Data mentah: 3628 baris
Kolom: ['app_name', 'package_name', 'review_id', 'user_name', 'review_text', 'rating', 'review_date']


## Sel 2 — Tahap 1: Deduplication (case-insensitive)

Duplikat dihapus dengan kunci (app_name, teks lowercase+strip) agar ulasan identik yang hanya berbeda kapitalisasi tidak terhitung ganda.

In [3]:
df['_tnorm'] = df['review_text'].astype(str).str.lower().str.strip()
n_dup = df.duplicated(subset=['app_name','_tnorm']).sum()
df = df.drop_duplicates(subset=['app_name','_tnorm']).drop(columns=['_tnorm']).reset_index(drop=True)
print(f"Duplikat dihapus: {n_dup}")
print(f"Sisa setelah dedup: {len(df)}")


Duplikat dihapus: 593
Sisa setelah dedup: 3035


## Sel 3 — Tahap 2: Deteksi Bahasa + Terjemahan

Metode: langdetect dengan seed dikunci (reproducible). Ulasan berbahasa Inggris diterjemahkan ke Indonesia memakai deep-translator, dan seluruh terjemahan dicatat pada translation_audit_log.csv.

In [4]:
from langdetect import detect, DetectorFactory, LangDetectException
DetectorFactory.seed = 0

def deteksi(t):
    try:
        if len(str(t).strip()) < 5: return 'id'
        l = detect(str(t))
        return l if l in ('id','en') else 'other'
    except LangDetectException:
        return 'id'

df['lang_detected'] = df['review_text'].apply(deteksi)
print("Distribusi deteksi bahasa (langdetect):")
print(df['lang_detected'].value_counts())


Distribusi deteksi bahasa (langdetect):
lang_detected
id       2709
other     295
en         31
Name: count, dtype: int64


In [5]:
# Terjemahkan hanya yang terdeteksi Inggris (audit log lengkap dengan app_name)
from deep_translator import GoogleTranslator

def translate_en(t):
    try:
        return GoogleTranslator(source='en', target='id').translate(str(t))
    except Exception:
        return t

mask_en = df['lang_detected'] == 'en'
log_tr = []
for idx in df[mask_en].index:
    asli  = df.at[idx, 'review_text']
    hasil = translate_en(asli)
    df.at[idx, 'review_text'] = hasil
    log_tr.append({
        'review_id':             df.at[idx, 'review_id'],
        'app_name':              df.at[idx, 'app_name'],       # ditambahkan agar audit informatif
        'rating':                df.at[idx, 'rating'],
        'teks_inggris_asli':     asli,
        'teks_hasil_terjemahan': hasil,
    })

audit = pd.DataFrame(log_tr)
audit.to_csv(f"{OUT_DIR}/translation_audit_log.csv", index=False, encoding='utf-8-sig')
print(f"Ulasan Inggris diterjemahkan : {mask_en.sum()}")
print(f"Baris audit log tersimpan    : {len(audit)}")
print(f"Kolom audit log              : {list(audit.columns)}")
print(f"File                         : {OUT_DIR}/translation_audit_log.csv")


Ulasan diterjemahkan: 31


## Sel 4 — Tahap 3-6: Case Folding, Noise Removal, Normalisasi Karakter Berulang

In [6]:
def step_lowercase(t): return str(t).lower()

def step_remove_noise(t):
    t = str(t)
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)   # URL
    t = re.sub(r'\S+@\S+\.\S+', ' ', t)             # email
    t = re.sub(r'[@#][\w]+', ' ', t)                  # mention/hashtag
    t = re.sub(r'[\U00010000-\U0010ffff]', ' ', t)   # emoji (supplementary)
    t = re.sub(r'[\u2600-\u27FF]', ' ', t)           # simbol/dingbats
    t = re.sub(r'[\u2B00-\u2BFF]', ' ', t)
    t = re.sub(r'[\uFE00-\uFE0F]', ' ', t)           # variation selectors
    t = re.sub(r'[\u200d\u200b\uFEFF]', ' ', t)      # zero-width
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def step_normalize_repeated(t):
    return re.sub(r'(.)\1{2,}', r'\1\1', str(t))   # 3+ berulang -> 2

print('Fungsi cleaning siap.')


Fungsi cleaning siap.


## Sel 5 — Tahap 7: Normalisasi Kata Tidak Baku + Protected Words

Kamus normalisasi dimuat dari kamuskatabaku.xlsx. Protected words melindungi nama merek dan istilah teknis (error, login, otp, dsb.) agar tidak ikut ternormalisasi.

In [7]:
# Kamus normalisasi
dk = pd.read_excel(f"{BASE}/02_preprocessing/data/kamuskatabaku.xlsx", sheet_name=0)
cols = dk.columns.tolist()
slang_dict = {str(k).lower().strip(): str(v).lower().strip()
              for k, v in dk[[cols[0], cols[1]]].dropna().values
              if str(k).strip() and str(v).strip()}
print(f"Kamus dimuat: {len(dk[[cols[0], cols[1]]].dropna())} entri ({len(slang_dict)} kunci unik)")

# Protected words: brand + istilah teknis aspek
PROTECTED_WORDS = {'wagely','paywatch','gajigesa','mekari','venteny','kasbon','ayokasbon',
                   'flex','hr','maintenance','update','login','logout','error','otp','pin',
                   'limit','terima','kasih','luar','karyawan'}

def step_normalize_slang(t, d=slang_dict, prot=PROTECTED_WORDS):
    out = []
    for w in str(t).split():
        wl = w.lower()
        out.append(wl if wl in prot else d.get(wl, wl))
    return ' '.join(out)


Kamus dimuat: 4975 entri (4952 kunci unik)


In [8]:
# Terapkan seluruh pipeline cleaning
def pipeline(t):
    t = step_lowercase(t)
    t = step_remove_noise(t)
    t = step_normalize_repeated(t)
    t = step_normalize_slang(t)
    return t

df['review_text_original'] = df['review_text']
df['review_text_clean'] = df['review_text'].apply(pipeline)
print("Contoh hasil cleaning:")
print(df[['review_text_original','review_text_clean']].head(3).to_string())


Contoh hasil cleaning:
                                                                                                                                                                                                                                  review_text_original                                                                                                                                                                                                                                             review_text_clean
0                                                                                                                                                                                  voucher indomaret sudah nggk pernah ada stok. gak jelas aplikasinya                                                                                                                                                                                        voucher indomaret sudah tidak pernah ada stok.

In [9]:
# Hapus baris yang teksnya kosong setelah cleaning
n_sebelum = len(df)
df = df[df['review_text_clean'].str.strip() != ''].reset_index(drop=True)
print(f"Baris kosong dihapus : {n_sebelum - len(df)}")
print(f"Sisa data bersih     : {len(df)}")


Baris kosong dihapus : 17
Sisa data bersih     : 3018


## Sel 6 — Tahap 8: Penghapusan Baris Kosong, Skip Logic & Simpan Data Bersih

- Baris yang teksnya kosong setelah cleaning dihapus.
- TERLALU_PENDEK: dua kata atau kurang. AMBIGU: seluruh katanya generik tanpa indikasi aspek.

In [10]:
KATA_GENERIK = {
    'bagus','baik','mantap','keren','oke','ok','sip','top','wow','wah',
    'ya','yep','yes','lumayan','alhamdulillah','luar biasa','sangat bagus',
    'sangat baik','cukup bagus','terima kasih','makasih','recommend',
    'recommended','good','great','nice','semoga lebih baik','terus berkembang',
    'sukses selalu','maju terus','semangat','keep it up',
}
KATA_PENILAIAN = {
    'bagus','baik','mantap','keren','oke','ok','sip','top','wow','wah',
    'lumayan','good','great','nice','sekali','banget','sangat','cukup',
    'agak','memang','sudah','udah','iya','sih','dong','lah'
}

def klasifikasi_skip(teks):
    t = str(teks).strip().lower()
    kata = t.split()
    if len(kata) < 3:
        return True, 'TERLALU_PENDEK'
    if t in KATA_GENERIK:
        return True, 'GENERIK'
    if len(kata) <= 5 and all(w in KATA_PENILAIAN for w in kata):
        return True, 'AMBIGU'
    return False, 'TIDAK_SKIP'

flags = df['review_text_clean'].apply(klasifikasi_skip)
df['skip_flag']     = [f[0] for f in flags]
df['skip_category'] = [f[1] for f in flags]

print("Distribusi skip:")
print(df[df['skip_flag']]['skip_category'].value_counts().to_string())
print(f"\nSiap anotasi (skip_flag=False): {(df['skip_flag']==False).sum()}")

cols_out = ['review_id','app_name','package_name','rating','review_date',
            'review_text_original','review_text_clean','skip_flag','skip_category']
df[cols_out].to_csv(f"{OUT_DIR}/dataset_clean_ewa_acsc.csv", index=False)
df[df['skip_flag']][['review_id','app_name','review_text_original','skip_category']].to_csv(
    f"{OUT_DIR}/skip_documentation.csv", index=False)
print("Disimpan: dataset_clean_ewa_acsc.csv, skip_documentation.csv")

Distribusi skip:
skip_category
TERLALU_PENDEK    365
AMBIGU              5

Siap anotasi (skip_flag=False): 2648
Disimpan: dataset_clean_ewa_acsc.csv, skip_documentation.csv


## Sel 7 — Contoh Transformasi Praproses (pipeline nyata)

In [11]:
contoh_asli = [
    "Ini aplikasinya bener bener bagus bangetttttt!",
    "Okeeeeey Sangat membantu para pekerja",
    "membantu buat belanja online.... makasi venteny",
    "gak bisa login error terus",
]
rows=[]
for t in contoh_asli:
    a = step_remove_noise(step_lowercase(t))
    b = step_normalize_repeated(a)
    c_ = step_normalize_slang(b)
    rows.append({"Teks Asli":t,"CaseFold+NoiseRemoval":a,"Norm.Karakter":b,"Teks Bersih":c_})
tabel46 = pd.DataFrame(rows)
import IPython.display as disp; disp.display(tabel46)
print("Transformasi dapat direproduksi: bangetttttt -> bangett, gak -> tidak; venteny/login/error dipertahankan (protected).")


                                         Teks Asli                            CaseFold+NoiseRemoval                                  Norm.Karakter                                           Teks Bersih
0   Ini aplikasinya bener bener bagus bangetttttt!   ini aplikasinya bener bener bagus bangetttttt!     ini aplikasinya bener bener bagus bangett!            ini aplikasinya benar benar bagus bangett!
1            Okeeeeey Sangat membantu para pekerja            okeeeeey sangat membantu para pekerja             okeey sangat membantu para pekerja                    okeey sangat membantu para pekerja
2  membantu buat belanja online.... makasi venteny  membantu buat belanja online.... makasi venteny  membantu buat belanja online.. makasi venteny  membantu untuk belanja online.. terima kasih venteny
3                       gak bisa login error terus                       gak bisa login error terus                     gak bisa login error terus                          tidak bisa login error t

## Sel 8 — EDA Distribusi Bahasa

Angka distribusi bahasa diambil dari langdetect (Sel 3) sebagai sumber resmi. Pendekatan leksikon dipakai untuk mengonfirmasi bahwa bahasa daerah dalam bentuk kalimat penuh = 0 (yang ada hanya serpihan alih kode). Sel ini menyimpan distribusi_bahasa.csv dan distribusi_bahasa.png.

In [12]:
# Distribusi bahasa: angka dari langdetect (Sel 3); leksikon dipakai untuk
# mengonfirmasi bahasa daerah kalimat penuh = 0.
import re, matplotlib.pyplot as plt

# 1) Angka resmi distribusi bahasa dari langdetect (kolom lang_detected, Sel 3)
dist_ld = df['lang_detected'].value_counts()
n_id    = int(dist_ld.get('id', 0))
n_en    = int(dist_ld.get('en', 0))
n_other = int(dist_ld.get('other', 0))
N       = len(df)
print("=== DISTRIBUSI BAHASA (langdetect - sumber resmi) ===")
print(f"  Indonesia               : {n_id:5d} ({n_id/N*100:.1f}%)")
print(f"  Lainnya (pendek/campur) : {n_other:5d} ({n_other/N*100:.1f}%)")
print(f"  Inggris (diterjemahkan) : {n_en:5d} ({n_en/N*100:.1f}%)")

# 2) Konfirmasi bahasa daerah KALIMAT PENUH via leksikon (bukan sekadar serpihan)
LEX_DAERAH = {
 'Jawa':  {'sing','ora','iki','iku','kowe','durung','piye','opo','isih','ojo','mboten','tenan','apik','matur','nuwun','suwun'},
 'Sunda': {'teu','abdi','anjeun','keur','kumaha','naon','hatur','nuhun','pisan','punten','mangga','sae','bade','jeung'},
 'Batak': {'molo','boi','adong','horas','mauliate','songon','godang','hami','hita','unang','nunga','denggan','roha'},
}
def daerah_kalimat_penuh(text):
    kata = re.findall(r'[a-z]+', str(text).lower())
    if len(kata) < 4:            # kalimat penuh minimal 4 kata
        return False
    for lex in LEX_DAERAH.values():
        # >=50% kata termasuk leksikon daerah -> dianggap kalimat penuh berbahasa daerah
        if sum(1 for w in kata if w in lex) >= max(2, len(kata)//2):
            return True
    return False

n_daerah_penuh = int(df['review_text_original'].apply(daerah_kalimat_penuh).sum())
print(f"  Bahasa daerah (kalimat penuh): {n_daerah_penuh:5d} ({n_daerah_penuh/N*100:.1f}%)")
print(f"  Total                   : {N} (100%)")

# 3) Simpan tabel distribusi bahasa sebagai bukti (untuk Tabel 4.9 dokumen)
tabel_bahasa = pd.DataFrame([
    {'bahasa':'Indonesia',                          'jumlah':n_id,          'persen':round(n_id/N*100,1)},
    {'bahasa':'Lainnya (teks pendek atau campuran)','jumlah':n_other,       'persen':round(n_other/N*100,1)},
    {'bahasa':'Inggris (diterjemahkan ke Indonesia)','jumlah':n_en,         'persen':round(n_en/N*100,1)},
    {'bahasa':'Bahasa daerah (kalimat penuh)',      'jumlah':n_daerah_penuh,'persen':round(n_daerah_penuh/N*100,1)},
])
tabel_bahasa.to_csv(f"{OUT_DIR}/distribusi_bahasa.csv", index=False, encoding='utf-8-sig')
print(f"\nTabel distribusi bahasa disimpan: {OUT_DIR}/distribusi_bahasa.csv")

# 4) Visualisasi (sumber gambar distribusi_bahasa.png)
warna = ['#2563eb','#9ca3af','#f59e0b','#dc2626']
fig, ax = plt.subplots(figsize=(8,4.6))
bar = ax.bar(tabel_bahasa['bahasa'], tabel_bahasa['jumlah'], color=warna)
for b,(_,row) in zip(bar, tabel_bahasa.iterrows()):
    ax.text(b.get_x()+b.get_width()/2, row['jumlah']+N*0.01,
            f"{int(row['jumlah'])}\n({row['persen']}%)", ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Jumlah Ulasan'); ax.set_ylim(0, N*1.02)
ax.set_title(f'Distribusi Bahasa Ulasan EWA (deteksi langdetect, N = {N})')
plt.xticks(rotation=8, ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/distribusi_bahasa.png", dpi=150)
plt.show()

# 5) Buang kolom bantu deteksi bahasa (tugasnya selesai)
df = df.drop(columns=['lang_detected'], errors='ignore')


=== DISTRIBUSI BAHASA (langdetect - sumber resmi) ===
  Indonesia               :  2709 (89.3%)
  Lainnya (pendek/campur) :   275 (9.7%)
  Inggris (diterjemahkan) :    31 (1.0%)
  Bahasa daerah (kalimat penuh):     0 (0.0%)
  Total                   : 3018 (100%)

Tabel distribusi bahasa disimpan: /content/drive/MyDrive/TA_ACSC_EWA/02_preprocessing/data


## Sel 9 — Analisis Near-Duplicate Pasca-Normalisasi

Setelah normalisasi, sejumlah ulasan berbeda menjadi berteks bersih identik. Analisis dilakukan di sini; penghapusan aktualnya dilakukan pada tahap finalisasi dataset (NB03) agar hasil anotasi terjaga.

In [13]:
siap = df[df['skip_flag']==False]
n_neardup = siap.duplicated(subset=['review_text_clean']).sum()
print(f"Kandidat anotasi: {len(siap)}")
print(f"Near-duplicate teks-bersih (review_id beda, teks sama): {n_neardup}")
print("\nContoh pasangan near-duplicate:")
dupmask = siap.duplicated(subset=['review_text_clean'], keep=False)
for t in siap[dupmask].sort_values('review_text_clean')['review_text_clean'].head(6):
    print(f"   '{t[:50]}'")
print("\nPenghapusan near-duplicate dilakukan pada tahap finalisasi dataset (NB03).")


Kandidat anotasi: 2648
Near-duplicate teks-bersih (review_id beda, teks sama): 46

Contoh pasangan near-duplicate:
   'aplikasi ini sangat membantu sekali'
   'aplikasi ini sangat membantu sekali'
   'aplikasi paling baik'
   'aplikasi paling baik'
   'aplikasi paling baik'
   'aplikasi paling baik'

Penghapusan near-duplicate dilakukan pada tahap finalisasi dataset (NB03).


## Sel 10 — Ringkasan Alur Data & Distribusi per Aplikasi


In [14]:
# Ringkasan alur angka dan distribusi per aplikasi pada data bersih
print("Alur data preprocessing:")
print(f"  Mentah                : 3628")
print(f"  Setelah dedup (-593)  : 3035")
print(f"  Setelah hapus kosong  : {len(df)}")
print(f"  Kandidat anotasi      : {(df['skip_flag']==False).sum()}")
print(f"  Skip                  : {df['skip_flag'].sum()} "
      f"({(df['skip_category']=='TERLALU_PENDEK').sum()} pendek + {(df['skip_category']=='AMBIGU').sum()} ambigu)")
print()
agg = df.groupby('app_name').agg(
    bersih=('review_id','count'),
    kandidat=('skip_flag', lambda s: int((~s).sum())),
    avg_rating=('rating','mean')).round(2)
print("Distribusi per aplikasi (data bersih):")
print(agg.to_string())


Alur data preprocessing:
  Mentah                : 3628
  Setelah dedup (-593)  : 3035
  Setelah hapus kosong  : 3018
  Kandidat anotasi      : 2648
  Skip                  : 370 (365 pendek + 5 ambigu)

Distribusi per aplikasi (data bersih):
             bersih  kandidat  avg_rating
app_name                                 
AyoKasbon        52        38        2.63
GajiGesa        423       387        3.50
Mekari Flex     795       653        3.66
Paywatch        241       206        4.48
VENTENY         653       629        4.38
Wagely          854       735        3.15


## Ringkasan

Pipeline preprocessing: deduplication (593 duplikat) -> deteksi bahasa langdetect + terjemahan (31 ulasan) -> cleaning -> normalisasi kamus dengan protected words -> penghapusan baris kosong -> skip logic. Hasil akhir: 3.018 ulasan bersih dan 2.648 kandidat anotasi.